# RNN Model for Question Answering

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import os
from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

## 1. Load and Clean Data

In [ ]:
# Define the relative path to the data
# Since the notebook is in 'model/' and the data is in 'data/', we use '../data/'
data_path = os.path.join("..", "data", "100_Unique_QA_Dataset.csv")

# Check if file exists, fallback to local directory if not
if not os.path.exists(data_path):
    data_path = "100_Unique_QA_Dataset.csv"

print(f"Loading data from: {data_path}")
df = pd.read_csv(data_path)

# Final cleanup to match requested format
df.columns = ['question', 'answer']
df.head()

## 2. Tokenization and Vocabulary

In [ ]:
def tokenizer(text):
    text = str(text).lower()
    text = text.replace("?", "").replace("'", "")
    return text.split()

# Build Vocabulary
vocab = {'<UNK>': 0}

def build_vocab(row):
    token_ans = tokenizer(row["question"])
    token_ques = tokenizer(row["answer"])
    merged_tkn = token_ans + token_ques

    for token in merged_tkn:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)

def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenizer(text):
        indexed_text.append(vocab.get(token, vocab["<UNK>"]))
    return indexed_text

# Test tokenizer and vocab
print(f"Vocab size: {len(vocab)}")
print(f"Sample indices for 'What is this?': {text_to_indices('What is this?', vocab)}")

## 3. Dataset and DataLoader

In [ ]:
class QADataloader(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self, index):
        num_ques = text_to_indices(self.df.iloc[index]['question'], self.vocab)
        num_ans = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
        return torch.tensor(num_ques), torch.tensor(num_ans)

def collate_fn(batch):
    questions, answers = zip(*batch)
    ques_padded = pad_sequence(questions, batch_first=True, padding_value=0)
    ans_padded = pad_sequence(answers, batch_first=True, padding_value=0)
    return ques_padded, ans_padded

dataset = QADataloader(df, vocab)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)

## 4. Model Definition

In [ ]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)

    def forward(self, question):
        embed_ques = self.embed(question)
        # rnn output: (output, hn)
        # output: [batch, seq_len, hidden_size]
        # hn: [num_layers, batch, hidden_size]
        output, hn = self.rnn(embed_ques)
        
        # We use the last hidden state for classification
        # For RNN with batch_first=True, hn is [1, batch, hidden_size]
        return self.fc(hn.squeeze(0))

## 5. Training Loop

In [ ]:
lr = 0.001
epochs = 20

model = SimpleRNN(len(vocab))
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

print("Starting training...")
for epoch in range(epochs):
    total_loss = 0
    for question, answer in dataloader:
        optimizer.zero_grad()
        
        # Forward pass
        pred = model(question) # Output shape: [batch_size, vocab_size]
        
        # Ensure answer is 1D (a list of single target indices)
        target = answer[:, 0].long()
        
        loss = criterion(pred, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch: {epoch+1}/{epochs} | Loss: {total_loss/len(dataloader):.4f}")

## 6. Prediction

In [ ]:
inv_vocab = {v: k for k, v in vocab.items()}

def predict(model, question, threshold=0.5):
    model.eval()
    num_ques = text_to_indices(question, vocab)
    torch_ques = torch.tensor(num_ques).unsqueeze(0)
    
    with torch.no_grad():
        output = model(torch_ques)
        probability = torch.nn.functional.softmax(output, dim=1)
        max_prob, index = torch.max(probability, dim=1)
        
    if max_prob < threshold:
        return "I don't know."
    
    return inv_vocab.get(index.item(), "<UNK>")

# Example prediction
print(f"Prediction for 'WHAT IS 5+5': {predict(model, 'WHAT IS 5+5')}")